In [ ]:
# ── COLAB SETUP ────────────────────────────────────────────────────────────────
# Run this cell first. It installs MDAnalysis and downloads the four simulation
# files this notebook is built around:
#   finalrun.gro     - GROMACS structure file (one frame, all atoms + coordinates)
#   finalrun.tpr     - GROMACS run/topology file (atoms, bonds, masses, box - binary)
#   finalrun.xtc     - GROMACS compressed trajectory (324 saved frames, every 20 ps)
#   finalrun.colvar  - PLUMED COLVAR file from an OPES multithermal simulation
#
# System: a VAL-PRO-TYR-LEU tetrapeptide in explicit water, run with GROMACS +
# PLUMED using OPES (On-the-fly Probability Enhanced Sampling), multithermal
# variant. numpy, matplotlib, and pandas are pre-installed in Google Colab.

import sys, os, subprocess, urllib.request

print(f"Python {sys.version}")
print("\nInstalling MDAnalysis (this may take ~60 s)...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "MDAnalysis"])
print("MDAnalysis installed ✓")

DATA_FILES = ["finalrun.gro", "finalrun.tpr", "finalrun.xtc", "finalrun.colvar"]
BASE_URL = ("https://raw.githubusercontent.com/jolayfield/chem-lab-tutorials/"
            "main/Track2_MDAnalysis/data/opes_tetrapeptide/")

print("\nFetching simulation files (finalrun.xtc is ~14 MB, may take a moment)...")
for fname in DATA_FILES:
    if os.path.exists(fname):
        print(f"  {fname} already present, skipping download")
        continue
    urllib.request.urlretrieve(BASE_URL + fname, fname)
    size_kb = os.path.getsize(fname) / 1024
    print(f"  {fname} downloaded ({size_kb:,.0f} KB)")

print("\nEnvironment ready ✓")


# Track 2 — OPES Multithermal Simulation of a Tetrapeptide
## MDAnalysis Meets PLUMED
### Demonstration Walkthrough

**Prerequisites:** *Part 1* and *Introduction to MDAnalysis* from this track. You should be
comfortable creating a `Universe`, selecting atoms, and looping over trajectory frames.
No physics or statistical-mechanics background is assumed beyond what you have seen in
general chemistry and organic chemistry (vectors, angles, Newman projections).

**The system:** a four-residue peptide, **Val–Pro–Tyr–Leu**, solvated in explicit
water and simulated with GROMACS. The simulation used **PLUMED's OPES (On-the-fly
Probability Enhanced Sampling) multithermal** method — an enhanced-sampling technique
that lets a single simulation behave as if it were run at many different temperatures at
once, so the peptide can escape energy traps that would trap it in an ordinary,
constant-temperature run.

You will work with **two independent records of the same simulation**:

| File | Written by | Contents |
|------|-----------|----------|
| `finalrun.gro` / `.tpr` / `.xtc` | GROMACS | Atomic **xyz coordinates** every 20 ps |
| `finalrun.colvar` | PLUMED | **Collective variables** (dihedral angles, radius of gyration, energy, bias) every 0.1 ps |

A major goal of this notebook is to show that these two files describe the *same physical
trajectory* — and to build enough understanding of the underlying coordinate geometry
that you could recompute PLUMED's numbers yourself, from nothing but xyz coordinates.

### Topics covered

| Section | Topic |
|---------|-------|
| 1 | Imports and meeting the system |
| 2 | Loading the trajectory: the `Universe` |
| 3 | Selecting atoms |
| 4 | Distance from xyz coordinates: Pythagoras and periodic boundaries |
| 5 | Radius of gyration and the "broken molecule" pitfall |
| 6 | Backbone dihedral angles: torsions and Newman projections |
| 7 | Reading the PLUMED COLVAR file |
| 8 | Cross-validating MDAnalysis against PLUMED |
| 9 | Visualizing the OPES multithermal landscape |

> **How to use this notebook:** This is a guided demonstration — every cell is
> filled in and ready to run. Work through it top to bottom with **Shift + Enter**.
> After each result, a commentary cell explains what the numbers mean and what to
> watch out for when you run the same analysis on your own simulations.


---
## Section 1 — Imports and Meeting the System

We need three things:
- **MDAnalysis** — reads GROMACS coordinate/trajectory files and gives us numpy arrays
- **numpy** — array math (already pre-installed in Colab)
- **matplotlib** and **pandas** — plotting and tabular data (also pre-installed)

The system is a designed tetrapeptide, **Val<sub>1</sub>–Pro<sub>2</sub>–Tyr<sub>3</sub>–Leu<sub>4</sub>**,
solvated in explicit water and simulated with an OPES multithermal bias acting on the
system's potential energy. That bias is what lets the peptide sample conformations it
would rarely reach in an unbiased simulation at a single temperature.

In [ ]:
import MDAnalysis as mda
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

print("MDAnalysis version:", mda.__version__)
print("numpy version:", np.__version__)


---
## Section 2 — Loading the Trajectory: The `Universe`

MDAnalysis organizes everything around a **`Universe`** object, built from two files:

- a **topology** file — tells MDAnalysis what atoms exist, their names, masses, and
  **bonds**. We use `finalrun.tpr`, GROMACS's binary run-input file, rather than the
  plain-text `.gro` file, because the `.gro` file stores only one frame of coordinates
  and does **not** store bond connectivity. You will see in Section 5 why bonds matter
  even for something as simple as computing a molecule's size.
- a **trajectory** file — the coordinates as a function of time. We use `finalrun.xtc`,
  GROMACS's compressed trajectory format.

```python
u = mda.Universe("finalrun.tpr", "finalrun.xtc")
```

In [ ]:
u = mda.Universe("finalrun.tpr", "finalrun.xtc")

print(f"Total atoms         : {u.atoms.n_atoms}")
print(f"Total residues       : {u.atoms.n_residues}")
print(f"Frames saved (xtc)   : {len(u.trajectory)}")
print(f"Time between frames  : {u.trajectory.dt} ps")
print(f"Total time covered   : {u.trajectory.totaltime} ps")

protein = u.select_atoms("protein")
print(f"\nProtein atoms: {protein.n_atoms}")
for res in protein.residues:
    print(f"  residue {res.resid}: {res.resname}  ({len(res.atoms)} atoms)")


#### Worked Example 2.1
Using the `Universe` object `u`, fill in:
- `sim_length_ns` — the total simulated time **in nanoseconds** (`u.trajectory.totaltime` is in ps; 1 ns = 1000 ps)
- `n_water_molecules` — the number of water molecules. Water is residue name `SOL` in
  this force field, and each water molecule has **3 atoms** (`OW`, `HW1`, `HW2`). Select
  all `SOL` atoms with `u.select_atoms("resname SOL")`, then divide the atom count by 3.
  Store the result as an **int**.

In [ ]:
# Worked Example 2.1
sim_length_ns = u.trajectory.totaltime / 1000

water_atoms = u.select_atoms("resname SOL")
n_water_molecules = int(water_atoms.n_atoms / 3)

print(f"Simulation length : {sim_length_ns:.2f} ns")
print(f"Water molecules   : {n_water_molecules}")


In [ ]:
# TEST 2.1
assert isinstance(sim_length_ns, float), "sim_length_ns should be a float"
assert abs(sim_length_ns - u.trajectory.totaltime / 1000) < 1e-6, "sim_length_ns conversion is off"
assert isinstance(n_water_molecules, int), "n_water_molecules should be an int"
assert n_water_molecules == u.select_atoms("resname SOL").n_atoms // 3, "n_water_molecules is incorrect"
print(f"✓  {sim_length_ns:.2f} ns simulated, {n_water_molecules} water molecules")


The peptide itself is tiny — 73 atoms — compared to the thousands of water atoms
surrounding it. That imbalance is normal for explicit-solvent simulations: most of the
computational cost goes into water, even though we usually only care about the solute.
This is also why almost every selection from here on starts with `protein` — without
it, every calculation would be swamped by 4,126 water molecules.

---
## Section 3 — Selecting Atoms

MDAnalysis's selection language reads almost like plain English. You have already seen
`"protein"`. A few more useful keywords:

| Selection | Meaning |
|-----------|---------|
| `"backbone"` | protein N, CA, C, O atoms only |
| `"name CA"` | atoms literally named `CA` (the alpha carbons) |
| `"resname TYR"` | all atoms of every tyrosine residue |
| `"resid 3"` | all atoms with residue number 3 |
| `"not name H*"` | exclude any atom whose name starts with `H` (removes hydrogens) |

Combine them with `and` / `or` / `not`.

In [ ]:
backbone = u.select_atoms("backbone")
ca_atoms = u.select_atoms("protein and name CA")
tyr = u.select_atoms("resname TYR")

print(f"Backbone atoms : {backbone.n_atoms}")
print(f"Cα atoms       : {ca_atoms.n_atoms}   (one per residue, confirming 4 residues)")
print(f"Cα residues    : {list(zip(ca_atoms.resids, ca_atoms.resnames))}")
print(f"Tyrosine atoms : {tyr.n_atoms}")


#### Worked Example 3.1
Create two `AtomGroup`s:
- `tyr_sidechain` — the tyrosine's **sidechain heavy atoms**: `resname TYR`, excluding
  backbone atoms (`not backbone`) and excluding hydrogens (`not name H*`)
- `protein_heavy` — all protein atoms that are **not hydrogen**

Then report how many atoms each contains.

In [ ]:
# Worked Example 3.1
tyr_sidechain = u.select_atoms("resname TYR and not backbone and not name H*")
protein_heavy = u.select_atoms("protein and not name H*")

print(f"Tyrosine sidechain heavy atoms : {tyr_sidechain.n_atoms}  {list(tyr_sidechain.names)}")
print(f"Protein heavy atoms (total)    : {protein_heavy.n_atoms}")


In [ ]:
# TEST 3.1
import MDAnalysis.core.groups
assert isinstance(tyr_sidechain, MDAnalysis.core.groups.AtomGroup), "tyr_sidechain should be an AtomGroup"
assert tyr_sidechain.n_atoms == 8, f"Expected 8 sidechain heavy atoms for TYR, got {tyr_sidechain.n_atoms}"
assert "OH" in tyr_sidechain.names, "tyr_sidechain should include the phenolic OH oxygen"
assert isinstance(protein_heavy, MDAnalysis.core.groups.AtomGroup), "protein_heavy should be an AtomGroup"
assert protein_heavy.n_atoms == 35, f"Expected 35 protein heavy atoms, got {protein_heavy.n_atoms}"
print(f"✓  TYR sidechain: {tyr_sidechain.n_atoms} heavy atoms | Protein heavy atoms: {protein_heavy.n_atoms}")


Notice tyrosine's sidechain includes its aromatic ring (`CG`, `CD1`, `CD2`, `CE1`,
`CE2`, `CZ`) plus the phenolic `OH` — the same ring you have drawn dozens of times in
organic chemistry, now as six atoms with real xyz coordinates. Out of 73 protein atoms,
only 35 are non-hydrogen — a useful reminder of how many atoms in a structure are simply
hydrogens, which is why "heavy-atom" selections are so common in structural analysis.

---
## Section 4 — Distance from xyz Coordinates: Pythagoras and Periodic Boundaries

Every `AtomGroup.positions` array is just a list of $(x, y, z)$ coordinates, in
ångströms. The distance between two atoms is the 3D generalization of the Pythagorean
theorem you know from geometry: for two points $\vec{r}_1=(x_1,y_1,z_1)$ and
$\vec{r}_2=(x_2,y_2,z_2)$,

$$d = \sqrt{(x_2-x_1)^2 + (y_2-y_1)^2 + (z_2-z_1)^2} = |\vec{r}_2 - \vec{r}_1|$$

In numpy this is `np.linalg.norm(p2 - p1)` or, written out, `np.sqrt(np.sum((p2-p1)**2))`.

**But there is a catch.** This simulation was run in a **periodic box** — GROMACS tiles
copies of the simulation cell infinitely in every direction so that no atom ever "hits a
wall." A molecule can drift out one face of the box and reappear on the opposite face.
When that happens, an atom's *stored* coordinates may be far from where the molecule
"really" is, and the naive distance formula above gives the wrong answer — sometimes by
tens of ångströms.

Also notice the box itself is not a simple cube. Check `u.dimensions`:
```
[Lx, Ly, Lz, alpha, beta, gamma] = [54.7, 54.7, 54.7, 70.5°, 109.5°, 70.5°]
```
Those non-90° angles mean this is a **truncated octahedron** box — a common, more
space-efficient shape for solvating a small, roughly spherical molecule than a cube,
since it needs less water to keep the same minimum peptide-to-image distance.

The fix is the **minimum-image convention**: instead of the raw coordinate difference,
use whichever periodic image of atom 2 is *closest* to atom 1. MDAnalysis does this for
you when you pass the box dimensions:
```python
from MDAnalysis.analysis.distances import dist
dist(group1, group2, box=u.dimensions)
```

In [ ]:
from MDAnalysis.analysis.distances import dist as mda_dist

ca1 = u.select_atoms("protein and name CA and resid 1")
ca4 = u.select_atoms("protein and name CA and resid 4")

def both_distances(frame):
    u.trajectory[frame]
    p1, p2 = ca1.positions[0], ca4.positions[0]
    d_naive = np.sqrt(np.sum((p2 - p1)**2))              # Pythagorean theorem, no PBC
    d_pbc = mda_dist(ca1, ca4, box=u.dimensions)[2][0]    # minimum-image convention
    return d_naive, d_pbc

print(f"Box: {u.dimensions}\n")
print(f"{'frame':>6}  {'naive (Å)':>11}  {'min-image (Å)':>14}")
for frame in [0, 285]:
    d_naive, d_pbc = both_distances(frame)
    print(f"{frame:>6}  {d_naive:11.3f}  {d_pbc:14.3f}")

print("\nAt frame 0 the two agree. At frame 285 the naive formula reports the")
print("peptide's two ends as 66 Å apart — further than the box is wide — because")
print("one Cα has been wrapped to the far side of the box.")


#### Worked Example 4.1
Loop over **every frame** of the trajectory and compute the Val1–Leu4 Cα–Cα distance
two ways: the naive Pythagorean formula, and the minimum-image (PBC-aware) formula.
Store:
- `dist_naive` — numpy array, one naive distance per frame (Å)
- `dist_pbc` — numpy array, one PBC-aware distance per frame (Å)

Then compute `n_frames_disagree` — the number of frames where the two methods differ by
more than 0.01 Å.

In [ ]:
# Worked Example 4.1
dist_naive_list = []
dist_pbc_list = []

for ts in u.trajectory:
    p1 = ca1.positions[0]
    p2 = ca4.positions[0]
    dist_naive_list.append(np.sqrt(np.sum((p2 - p1)**2)))
    dist_pbc_list.append(mda_dist(ca1, ca4, box=ts.dimensions)[2][0])

dist_naive = np.array(dist_naive_list)
dist_pbc = np.array(dist_pbc_list)
n_frames_disagree = int(np.sum(np.abs(dist_naive - dist_pbc) > 0.01))

print(f"Frames where naive and PBC-aware distance disagree by >0.01 Å: "
      f"{n_frames_disagree} / {len(u.trajectory)}")
print(f"Largest disagreement: {np.max(np.abs(dist_naive - dist_pbc)):.2f} Å")


In [ ]:
# TEST 4.1
assert isinstance(dist_naive, np.ndarray) and len(dist_naive) == len(u.trajectory), "dist_naive should have one value per frame"
assert isinstance(dist_pbc, np.ndarray) and len(dist_pbc) == len(u.trajectory), "dist_pbc should have one value per frame"
assert isinstance(n_frames_disagree, int), "n_frames_disagree should be an int"
assert n_frames_disagree > 0, "You should find at least some disagreement in this trajectory"
assert n_frames_disagree < len(u.trajectory), "Not every frame should disagree"
print(f"✓  {n_frames_disagree} of {len(u.trajectory)} frames disagree by >0.01 Å "
      f"(max difference {np.max(np.abs(dist_naive - dist_pbc)):.1f} Å)")

plt.figure(figsize=(7, 3))
plt.plot(dist_naive, label="naive (no PBC)", alpha=0.7)
plt.plot(dist_pbc, label="minimum image (PBC)", alpha=0.7)
plt.xlabel("Frame")
plt.ylabel("Val1(Cα)–Leu4(Cα) distance (Å)")
plt.legend()
plt.tight_layout()
plt.show()


In this trajectory, the naive and PBC-aware distances disagree in roughly one frame in
five — and when they disagree, it is not by a rounding error, it is by tens of
ångströms, because one atom's raw coordinate has been wrapped into a neighboring
periodic image. This is not a hypothetical warning from a textbook: it is something that
actually happens in this exact dataset, and it would silently corrupt any distance-based
analysis (contacts, hydrogen bonds, end-to-end distance) that forgot to pass `box=...`.

---
## Section 5 — Radius of Gyration, and the "Broken Molecule" Pitfall

The **radius of gyration** $R_g$ is a single number describing how spread out a molecule
is. It is the mass-weighted root-mean-square distance of every atom from the molecule's
center of mass. Small $R_g$ means compact and folded; large $R_g$ means extended.

**Step 1 — center of mass.** The mass-weighted average position:

$$\vec{r}_{\text{CoM}} = \frac{\sum_i m_i \vec{r}_i}{\sum_i m_i}$$

**Step 2 — mass-weighted mean squared distance from that center:**

$$R_g = \sqrt{\frac{\sum_i m_i \, |\vec{r}_i - \vec{r}_{\text{CoM}}|^2}{\sum_i m_i}}$$

Every symbol here you can compute from the xyz coordinates and masses MDAnalysis already
gives you. `AtomGroup.radius_of_gyration()` does exactly this, and in Practice 5.1 you
will reproduce it by hand.

### The pitfall

$R_g$ is where periodic boundaries bite hardest. If GROMACS wrapped the peptide so that
part of it sits at one edge of the box and part at the opposite edge, the molecule looks
**shattered across the box** even though it is physically intact. The center of mass
lands somewhere in the middle of the box — nowhere near any actual atom — and $R_g$
comes out enormous.

The fix is `AtomGroup.unwrap(compound='fragments')`. A **fragment** is a set of atoms
connected by bonds — MDAnalysis walks the bond network and shifts periodic images so that
bonded atoms stay next to each other. This is exactly why we loaded the `.tpr` file
instead of the `.gro` file back in Section 2: **`.gro` files carry no bond information**,
so `unwrap()` would have nothing to work with.

In [ ]:
# Compare Rg before and after unwrapping, on a frame where it matters.

u.trajectory[0]
rg_broken = protein.radius_of_gyration()

protein.unwrap(compound='fragments')
rg_fixed = protein.radius_of_gyration()

print(f"Rg at frame 0, as stored in the file : {rg_broken:.3f} Å")
print(f"Rg at frame 0, after unwrap()        : {rg_fixed:.3f} Å")
print()
print(f"The peptide is really about {rg_fixed/10:.3f} nm across — a compact tetrapeptide.")
print(f"The unwrapped value is {rg_broken/rg_fixed:.1f}x too large: the molecule was")
print("split across the periodic box boundary.")


#### Worked Example 5.1
Write the radius of gyration **by hand**, from coordinates and masses only — no
`radius_of_gyration()` call — and verify it matches MDAnalysis.

Complete the function `manual_rg(atomgroup)` so that it:
1. gets `pos = atomgroup.positions` (an $N \times 3$ array) and `m = atomgroup.masses` (length $N$)
2. computes the center of mass. Hint: `(pos * m[:, None]).sum(axis=0) / m.sum()`.
   The `m[:, None]` reshapes the mass array so each mass multiplies a whole xyz row.
3. computes squared distances of each atom from that center: `((pos - com)**2).sum(axis=1)`
4. returns `np.sqrt((m * sq_dists).sum() / m.sum())`

Then set `rg_manual` and `rg_library` for frame 0 (**after** unwrapping).

In [ ]:
# Worked Example 5.1
def manual_rg(atomgroup):
    pos = atomgroup.positions
    m = atomgroup.masses
    com = (pos * m[:, None]).sum(axis=0) / m.sum()
    sq_dists = ((pos - com)**2).sum(axis=1)
    return np.sqrt((m * sq_dists).sum() / m.sum())

u.trajectory[0]
protein.unwrap(compound='fragments')

rg_manual = manual_rg(protein)
rg_library = protein.radius_of_gyration()

print(f"Rg computed by hand      : {rg_manual:.6f} Å")
print(f"Rg from MDAnalysis        : {rg_library:.6f} Å")
print(f"Difference                : {abs(rg_manual - rg_library):.2e} Å")


In [ ]:
# TEST 5.1
assert rg_manual is not None and rg_library is not None, "Set both rg_manual and rg_library"
assert abs(rg_manual - rg_library) < 1e-4, (
    f"Your manual Rg ({rg_manual}) does not match MDAnalysis ({rg_library}). "
    "Check that you weighted by mass in BOTH the center-of-mass and the mean-square step.")
assert 3.0 < rg_manual < 6.0, f"Rg should be ~4.3 Å for this tetrapeptide, got {rg_manual}"
print(f"✓  Manual Rg = {rg_manual:.4f} Å matches MDAnalysis ({rg_library:.4f} Å)")


Getting the same number two different ways is the point of this exercise. The formula
is not a black box: it is a weighted average you could do on paper for four atoms, just
applied to 73. Note also that mass-weighting matters — hydrogens are 1 amu against
carbon's 12, so they contribute very little to $R_g$ even though they are half the atoms.

#### Worked Example 5.2
Now build the $R_g$ **time series** over the whole trajectory, computing it both ways so
you can see how often the PBC artifact strikes.

For each frame, record:
- the raw $R_g$ as stored in the file (before unwrapping)
- the $R_g$ after `protein.unwrap(compound='fragments')`

Store as `rg_raw` and `rg_unwrapped` (numpy arrays, in Å), and count
`n_broken_frames` — frames where they differ by more than 0.5 Å.

> **Order matters:** compute the raw value *first*, then unwrap, then compute the
> corrected value — `unwrap()` modifies the coordinates in place for that frame.

In [ ]:
# Worked Example 5.2
rg_raw_list = []
rg_unwrapped_list = []

for ts in u.trajectory:
    rg_raw_list.append(protein.radius_of_gyration())      # BEFORE unwrap
    protein.unwrap(compound='fragments')
    rg_unwrapped_list.append(protein.radius_of_gyration())  # AFTER unwrap

rg_raw = np.array(rg_raw_list)
rg_unwrapped = np.array(rg_unwrapped_list)
n_broken_frames = int(np.sum(np.abs(rg_raw - rg_unwrapped) > 0.5))

print(f"Frames with a broken-molecule artifact: {n_broken_frames} / {len(u.trajectory)}")
print(f"Corrected Rg: mean {rg_unwrapped.mean():.2f} Å, "
      f"range {rg_unwrapped.min():.2f}–{rg_unwrapped.max():.2f} Å")


In [ ]:
# TEST 5.2
assert isinstance(rg_unwrapped, np.ndarray) and len(rg_unwrapped) == len(u.trajectory), "rg_unwrapped should have one value per frame"
assert isinstance(rg_raw, np.ndarray) and len(rg_raw) == len(u.trajectory), "rg_raw should have one value per frame"
assert np.all(rg_unwrapped < 10.0), "Unwrapped Rg should stay under 10 Å for this tetrapeptide"
assert n_broken_frames > 50, "You should find many broken frames — did you compute rg_raw BEFORE unwrapping?"
print(f"✓  {n_broken_frames} of {len(u.trajectory)} frames needed unwrapping; "
      f"corrected Rg = {rg_unwrapped.mean():.2f} ± {rg_unwrapped.std():.2f} Å")

plt.figure(figsize=(7, 3))
plt.plot(rg_raw, '.', ms=3, alpha=0.5, label="raw (broken molecules)")
plt.plot(rg_unwrapped, '-', lw=1, label="unwrapped (correct)")
plt.xlabel("Frame")
plt.ylabel("Radius of gyration (Å)")
plt.legend()
plt.tight_layout()
plt.show()


Nearly half the frames in this trajectory are affected. On the plot, the corrected
$R_g$ is a tight band around 4–5 Å, while the raw values scatter up to sevenfold higher
— those are the frames where the peptide happened to straddle a box face when GROMACS
wrote the coordinates. Any analysis that skipped `unwrap()` would report a peptide that
appears to unfold and refold dramatically every few frames. That is a plotting artifact,
not chemistry, and it is the single most common mistake in beginner trajectory analysis.

---
## Section 6 — Backbone Dihedral Angles: Torsions and Newman Projections

You have already met dihedral angles in organic chemistry, under a different name.
When you drew a **Newman projection** of butane and labeled conformations *anti*,
*gauche*, and *eclipsed*, you were describing a **dihedral angle** — the angle of
rotation about a central bond.

A dihedral is defined by **four** atoms in sequence, $A-B-C-D$. Sight down the central
$B \to C$ bond (exactly what a Newman projection does: $B$ is the front atom, $C$ is the
back atom). The dihedral angle is the angle between the projection of $A$ and the
projection of $D$ onto the plane perpendicular to $B-C$.

### The protein backbone: φ and ψ

A protein backbone repeats the pattern $\cdots N - C_\alpha - C - N - C_\alpha - C \cdots$.
Two rotatable dihedrals define each residue's local shape:

| Angle | Four atoms | Rotation about |
|-------|-----------|----------------|
| $\phi_i$ (phi) | $C_{i-1} - N_i - C_{\alpha,i} - C_i$ | the $N-C_\alpha$ bond |
| $\psi_i$ (psi) | $N_i - C_{\alpha,i} - C_i - N_{i+1}$ | the $C_\alpha - C$ bond |

The third backbone dihedral, $\omega$ (the peptide bond itself), is essentially locked
near 180° by the amide bond's partial double-bond character — the resonance you learned
in organic chemistry, showing up as a geometric constraint.

Note the **edge effects**: residue 1 has no preceding residue, so it has no $\phi_1$;
residue 4 has no following residue, so it has no $\psi_4$. For our four-residue peptide
that leaves exactly **six** backbone dihedrals: $\psi_1, \phi_2, \psi_2, \phi_3, \psi_3, \phi_4$.
Those are precisely the six angles PLUMED tracked in the COLVAR file — you will see them
by name in Section 7.

### The formula

Given four position vectors, build three bond vectors:

$$\vec{b}_1 = \vec{r}_A - \vec{r}_B, \qquad \vec{b}_2 = \vec{r}_C - \vec{r}_B, \qquad \vec{b}_3 = \vec{r}_D - \vec{r}_C$$

Let $\hat{b}_2 = \vec{b}_2 / |\vec{b}_2|$ be the unit vector along the central bond — the
axis we sight down. Project $\vec{b}_1$ and $\vec{b}_3$ into the plane perpendicular to it
by subtracting off their components along the axis:

$$\vec{v} = \vec{b}_1 - (\vec{b}_1 \cdot \hat{b}_2)\,\hat{b}_2, \qquad
  \vec{w} = \vec{b}_3 - (\vec{b}_3 \cdot \hat{b}_2)\,\hat{b}_2$$

These two in-plane vectors are exactly the front and back bonds you draw in a Newman
projection. The angle between them is the dihedral. Using `arctan2` rather than `arccos`
gives the correct **sign** and full $-\pi$ to $+\pi$ range:

$$x = \vec{v} \cdot \vec{w}, \qquad y = (\hat{b}_2 \times \vec{v}) \cdot \vec{w}, \qquad
  \theta = \operatorname{atan2}(y,\, x)$$

The result is in **radians**, which is also how PLUMED reports dihedrals — convenient for
the comparison in Section 8.

In [ ]:
# Compute psi_1 (N1 - CA1 - C1 - N2) at frame 0, by hand and with MDAnalysis.

from MDAnalysis.lib.distances import calc_dihedrals

u.trajectory[0]
protein.unwrap(compound='fragments')

res1 = u.select_atoms("protein and resid 1")
res2 = u.select_atoms("protein and resid 2")

N1 = res1.select_atoms("name N").positions[0]
CA1 = res1.select_atoms("name CA").positions[0]
C1 = res1.select_atoms("name C").positions[0]
N2 = res2.select_atoms("name N").positions[0]

# --- by hand, following the formula above ---
b1 = N1 - CA1          # r_A - r_B
b2 = C1 - CA1          # r_C - r_B
b3 = N2 - C1           # r_D - r_C

b2_hat = b2 / np.linalg.norm(b2)
v = b1 - np.dot(b1, b2_hat) * b2_hat
w = b3 - np.dot(b3, b2_hat) * b2_hat

x = np.dot(v, w)
y = np.dot(np.cross(b2_hat, v), w)
psi1_manual = np.arctan2(y, x)

# --- with MDAnalysis ---
psi1_library = calc_dihedrals(N1, CA1, C1, N2)

print(f"psi_1 by hand      : {psi1_manual:.6f} rad  ({np.degrees(psi1_manual):.2f}°)")
print(f"psi_1 by MDAnalysis: {psi1_library:.6f} rad  ({np.degrees(psi1_library):.2f}°)")


#### Worked Example 6.1
Package the dihedral calculation into a reusable function, then use it.

Complete `manual_dihedral(pA, pB, pC, pD)` following the formula in the text above — it
should take four position arrays of shape `(3,)` and return the angle in **radians**.

Then use it to compute `phi2_manual`, the $\phi$ angle of residue 2 (proline), defined
by the four atoms $C_1 - N_2 - C_{\alpha,2} - C_2$. Note the atoms span two residues:
the carbonyl carbon `C` of residue 1, then `N`, `CA`, `C` of residue 2.

Also set `phi2_library` using `calc_dihedrals` on the same four positions.

In [ ]:
# Worked Example 6.1
def manual_dihedral(pA, pB, pC, pD):
    """Dihedral angle A-B-C-D in radians, in the range -pi to pi."""
    b1 = pA - pB
    b2 = pC - pB
    b3 = pD - pC

    b2_hat = b2 / np.linalg.norm(b2)
    v = b1 - np.dot(b1, b2_hat) * b2_hat
    w = b3 - np.dot(b3, b2_hat) * b2_hat

    x = np.dot(v, w)
    y = np.dot(np.cross(b2_hat, v), w)
    return np.arctan2(y, x)

u.trajectory[0]
protein.unwrap(compound='fragments')

C1 = u.select_atoms("protein and resid 1 and name C").positions[0]
N2 = u.select_atoms("protein and resid 2 and name N").positions[0]
CA2 = u.select_atoms("protein and resid 2 and name CA").positions[0]
C2 = u.select_atoms("protein and resid 2 and name C").positions[0]

phi2_manual = manual_dihedral(C1, N2, CA2, C2)
phi2_library = calc_dihedrals(C1, N2, CA2, C2)

print(f"phi_2 by hand      : {phi2_manual:.6f} rad  ({np.degrees(phi2_manual):.2f}°)")
print(f"phi_2 by MDAnalysis: {phi2_library:.6f} rad  ({np.degrees(phi2_library):.2f}°)")


In [ ]:
# TEST 6.1
assert phi2_manual is not None and phi2_library is not None, "Set both phi2_manual and phi2_library"
assert abs(phi2_manual - phi2_library) < 1e-5, (
    f"Your manual dihedral ({phi2_manual}) disagrees with MDAnalysis ({phi2_library}). "
    "If the magnitude matches but the sign is flipped, check the direction of b1: "
    "it should be pA - pB, not pB - pA.")
assert -np.pi <= phi2_manual <= np.pi, "Dihedral should be in the range -pi to pi"
assert abs(phi2_manual - 1.3742) < 0.01, (
    f"Expected phi_2 ≈ 1.374 rad at frame 0, got {phi2_manual:.4f}. "
    "If you got about -0.83, you forgot to unwrap() before reading positions — "
    "the peptide is split across the periodic box at frame 0.")
print(f"✓  phi_2 (Pro) = {phi2_manual:.4f} rad = {np.degrees(phi2_manual):.1f}°, matches MDAnalysis")


Two traps are baked into this exercise, and the test messages name both.

**The sign convention.** Using `arccos` instead of `arctan2` would give the right
magnitude but throw away the sign, collapsing +60° and −60° into the same value — which
would make *gauche⁺* and *gauche⁻* indistinguishable. Likewise, defining `b1` as
`pB - pA` instead of `pA - pB` flips the sign of every angle you compute.

**Unwrapping again.** If you skip `protein.unwrap(compound='fragments')` at frame 0, this
dihedral comes out as −0.83 rad instead of +1.374 rad — not a small error, a completely
different conformation. The reason is the same PBC artifact from Section 5: at frame 0
the peptide straddles a box face, so one of the four atoms has coordinates from a
neighboring periodic image. Section 5's lesson was not a one-off caveat; it applies to
*every* geometric quantity you compute from this trajectory.

#### Worked Example 6.2
Now compute all **six** backbone dihedrals over the entire trajectory, so we can compare
them to PLUMED in Section 8.

The six angles, with their defining atoms as `(resid, atomname)` pairs, are already
listed for you in `DIHEDRAL_DEFS`. Your job is to loop over frames and fill
`dihedrals_mda`, a dictionary mapping each angle name to a numpy array of one value per
frame (radians). Also build `times_mda`, the simulation time in ps for each frame.

For each frame: unwrap first, then for each of the six definitions, pull the four
positions and call `calc_dihedrals`.

> **Efficiency tip:** select the `AtomGroup`s **once, before** the frame loop.
> `positions` updates automatically as the trajectory advances, so re-selecting inside
> the loop just wastes time.

In [ ]:
# Worked Example 6.2
DIHEDRAL_DEFS = {
    "psi1": [(1, "N"), (1, "CA"), (1, "C"), (2, "N")],
    "phi2": [(1, "C"), (2, "N"), (2, "CA"), (2, "C")],
    "psi2": [(2, "N"), (2, "CA"), (2, "C"), (3, "N")],
    "phi3": [(2, "C"), (3, "N"), (3, "CA"), (3, "C")],
    "psi3": [(3, "N"), (3, "CA"), (3, "C"), (4, "N")],
    "phi4": [(3, "C"), (4, "N"), (4, "CA"), (4, "C")],
}

# Select the atom groups once, outside the loop
dihedral_groups = {
    name: [u.select_atoms(f"protein and resid {r} and name {a}") for r, a in atoms]
    for name, atoms in DIHEDRAL_DEFS.items()
}

times_mda_list = []
dihedrals_mda = {name: [] for name in DIHEDRAL_DEFS}

for ts in u.trajectory:
    protein.unwrap(compound='fragments')
    times_mda_list.append(ts.time)
    for name, groups in dihedral_groups.items():
        pA, pB, pC, pD = [g.positions[0] for g in groups]
        dihedrals_mda[name].append(calc_dihedrals(pA, pB, pC, pD))

times_mda = np.array(times_mda_list)
dihedrals_mda = {name: np.array(vals) for name, vals in dihedrals_mda.items()}

for name, vals in dihedrals_mda.items():
    print(f"{name}: {len(vals)} values, mean {np.degrees(vals.mean()):7.1f}°")


In [ ]:
# TEST 6.2
assert isinstance(dihedrals_mda, dict), "dihedrals_mda should be a dictionary"
assert set(dihedrals_mda.keys()) == {"psi1", "phi2", "psi2", "phi3", "psi3", "phi4"}, \
    f"Expected the six backbone dihedrals, got {sorted(dihedrals_mda.keys())}"
for name, vals in dihedrals_mda.items():
    assert len(vals) == len(u.trajectory), f"{name} should have one value per frame"
    assert np.all(np.abs(vals) <= np.pi + 1e-6), f"{name} values should be radians in -pi..pi"
assert len(times_mda) == len(u.trajectory), "times_mda length mismatch"
assert times_mda[0] == 0.0 and times_mda[-1] == 6460.0, "times_mda should run 0 to 6460 ps"
print(f"✓  Six dihedrals computed over {len(times_mda)} frames "
      f"({times_mda[0]:.0f}–{times_mda[-1]:.0f} ps)")

fig, axes = plt.subplots(3, 2, figsize=(10, 6), sharex=True)
for ax, (name, vals) in zip(axes.flat, dihedrals_mda.items()):
    ax.plot(times_mda / 1000, np.degrees(vals), '.', ms=2)
    ax.set_ylabel(f"{name} (°)")
    ax.set_ylim(-180, 180)
for ax in axes[-1]:
    ax.set_xlabel("Time (ns)")
plt.tight_layout()
plt.show()


Look at the six panels together and compare how *wide* each one is.

Proline's **φ₂ is the tightest of the six** — its middle 50% of values spans only about
20°, against 173° for ψ₃. That is the five-membered ring at work: proline's sidechain
loops back and bonds to its own backbone nitrogen, mechanically restricting rotation
about the N–Cα bond. You are reading a piece of organic chemistry directly off the
geometry, and it is why proline so often anchors turns in peptides.

**ψ₃**, by contrast, jumps between two well-separated bands. Each band is a distinct
**rotamer** — a staggered conformation the molecule prefers, separated from its neighbor
by an eclipsed barrier, exactly as in the butane energy diagram. Those jumps are the
rare, interesting events, and OPES multithermal sampling exists precisely to make them
happen more often than they would naturally.

---
## Section 7 — Reading the PLUMED COLVAR File

While GROMACS was writing coordinates, **PLUMED** was running alongside it computing
**collective variables** (CVs) — a handful of numbers that summarize the peptide's shape
at each instant — and writing them to `finalrun.colvar`.

A COLVAR file is a plain-text table with a special header. The first line names the
columns:

```
#! FIELDS time ene psi1 phi2 psi2 phi3 psi3 phi4 rg ecv.ene opes.bias
```

Subsequent `#! SET` lines record metadata, such as the periodic domain of each angle
(`min_psi1 -pi`, `max_psi1 pi`). Everything after that is numeric data, one row per
PLUMED step.

| Column | Meaning | Units |
|--------|---------|-------|
| `time` | simulation time | ps |
| `ene` | total potential energy of the system | kJ/mol |
| `psi1`…`phi4` | the six backbone dihedrals — **the same six you just computed** | radians |
| `rg` | radius of gyration of the peptide | **nm** |
| `ecv.ene` | the energy as seen by the OPES expansion CV | kJ/mol |
| `opes.bias` | the bias potential OPES has deposited so far | kJ/mol |

**Two unit traps to watch:**
1. PLUMED works in **nanometers**; MDAnalysis works in **ångströms**. 1 nm = 10 Å.
2. PLUMED reports dihedrals in **radians**, same as `calc_dihedrals` — no conversion
   needed there.

**A sampling-rate trap:** PLUMED wrote a row every **0.1 ps**, while GROMACS saved a
trajectory frame only every **20 ps**. So the COLVAR file has ~65,000 rows against the
trajectory's 324 frames — 200× more time resolution. To compare them we will have to
match on the `time` column, not on row index.

### What OPES multithermal is doing

An ordinary MD simulation at 300 K samples conformations weighted by the Boltzmann
distribution at that one temperature. Barriers much larger than $k_B T$ are essentially
never crossed, so a small peptide can sit in one conformation for the whole run.

**OPES multithermal** adds a bias potential that is a function of the system's potential
energy, constructed so the simulation samples a *broadened* energy distribution — as if
it were simultaneously at a range of temperatures. High-energy configurations (barrier
tops) become far more likely, so the peptide crosses barriers frequently, while the
`opes.bias` column records exactly how much artificial help it received at each step.
That record is what makes it possible to reweight back to unbiased 300 K behavior.

In [ ]:
# Look at the raw file before parsing it.

with open("finalrun.colvar") as f:
    for i, line in enumerate(f):
        if i < 3 or (13 <= i <= 15):
            print(repr(line.rstrip()))
        if i > 15:
            break


#### Worked Example 7.1
Parse the COLVAR file into a pandas DataFrame.

1. Open the file and read the **first line**. Strip off the leading `#! FIELDS ` and
   `.split()` the rest to get the column names as a list. Store as `colvar_columns`.
2. Read the numeric data with
   `pd.read_csv("finalrun.colvar", sep=r"\s+", comment="#", names=colvar_columns)`.
   The `comment="#"` argument skips every header line; `sep=r"\s+"` splits on runs of
   whitespace. Store the result as `colvar_raw`.
3. This simulation was stopped while PLUMED was mid-write, so the **very last line of
   the file is incomplete** and parses into a row of `NaN`s. Drop it:
   `colvar = colvar_raw.dropna().reset_index(drop=True)`. Set `n_dropped` to how many
   rows this removed.
4. Set `colvar_dt` to the time step between consecutive PLUMED rows, in ps.

In [ ]:
# Worked Example 7.1
with open("finalrun.colvar") as f:
    header = f.readline()

colvar_columns = header.replace("#! FIELDS ", "").split()

colvar_raw = pd.read_csv("finalrun.colvar", sep=r"\s+", comment="#", names=colvar_columns)

colvar = colvar_raw.dropna().reset_index(drop=True)
n_dropped = len(colvar_raw) - len(colvar)

colvar_dt = colvar["time"].iloc[1] - colvar["time"].iloc[0]

print("Columns:", colvar_columns)
print(f"Rows parsed: {len(colvar_raw):,}  ->  {len(colvar):,} after dropping {n_dropped} incomplete row(s)")
print(f"PLUMED time step: {colvar_dt} ps")
print(f"Time range: {colvar['time'].min():.1f} – {colvar['time'].max():.1f} ps")
print()
print(colvar.head())


In [ ]:
# TEST 7.1
assert isinstance(colvar_columns, list), "colvar_columns should be a list"
assert colvar_columns[0] == "time", f"First column should be 'time', got {colvar_columns[0]!r}"
assert "opes.bias" in colvar_columns, "'opes.bias' should be among the columns"
assert len(colvar_columns) == 11, f"Expected 11 columns, got {len(colvar_columns)}"
assert isinstance(colvar, pd.DataFrame), "colvar should be a pandas DataFrame"
assert len(colvar) == 65133, f"Expected 65,133 complete rows, got {len(colvar)}"
assert n_dropped == 1, f"Exactly one incomplete row should have been dropped, got {n_dropped}"
assert colvar.isna().sum().sum() == 0, "colvar should contain no NaN values after dropping"
assert abs(colvar_dt - 0.1) < 1e-6, f"PLUMED time step should be 0.1 ps, got {colvar_dt}"
print(f"✓  Parsed {len(colvar):,} PLUMED rows ({n_dropped} incomplete row dropped), dt = {colvar_dt} ps")


Two things worth noticing here.

First, the scale difference: 65,133 PLUMED rows against 324 trajectory frames. PLUMED
computes its collective variables at essentially every MD step because it needs them to
apply the bias in real time, whereas writing full coordinates that often would produce an
enormous file for little benefit. This is the normal arrangement in enhanced sampling:
cheap CVs at high frequency, expensive coordinates at low frequency.

Second, that truncated final line is what real simulation data looks like. The job hit
its wall-clock limit partway through writing a row, and the file was left mid-number.
`dropna()` handles it in one line — but only if you *look* first. A silent `NaN`
propagating into a mean or a plot is much harder to notice later.

#### Worked Example 7.2 — A real data artifact

Look at the `ene` column. Something is wrong with it. Investigate:

- `n_zero_ene` — how many rows have `ene` **exactly** equal to 0.0
- `frac_zero` — that count as a fraction of all rows
- `zero_rows_alternate` — a bool: check whether the zeros fall on *alternating* rows.
  Test this by taking the first 100 rows, and verifying that every row with an odd index
  has `ene == 0` while every row with an even index does not.

A potential energy of *exactly* zero is physically impossible for a solvated peptide —
this is a logging artifact, not chemistry.

In [ ]:
# Worked Example 7.2
n_zero_ene = int((colvar["ene"] == 0.0).sum())
frac_zero = n_zero_ene / len(colvar)

first100 = colvar["ene"].iloc[:100].values
odd_all_zero = np.all(first100[1::2] == 0.0)
even_none_zero = np.all(first100[0::2] != 0.0)
zero_rows_alternate = bool(odd_all_zero and even_none_zero)

print(f"Rows with ene exactly 0.0 : {n_zero_ene:,} of {len(colvar):,} ({frac_zero:.1%})")
print(f"Zeros fall on alternating rows: {zero_rows_alternate}")
print()
print("First 6 rows of time and ene:")
print(colvar[["time", "ene"]].head(6).to_string(index=False))


In [ ]:
# TEST 7.2
assert isinstance(n_zero_ene, int), "n_zero_ene should be an int"
assert n_zero_ene == 32566, f"Expected 32,566 zero-energy rows, got {n_zero_ene}"
assert 0.45 < frac_zero < 0.55, f"Expected roughly half the rows, got {frac_zero:.1%}"
assert zero_rows_alternate is True, "The zeros should fall on alternating rows"
print(f"✓  Found the artifact: {n_zero_ene:,} rows ({frac_zero:.1%}) have ene exactly 0.0, "
      "on alternating rows")


This is a genuine, well-known GROMACS/PLUMED interaction, not something manufactured
for the exercise. GROMACS only computes the full potential energy on certain steps
(controlled by `nstcalcenergy`), and on the intervening steps PLUMED's `ENERGY` action
receives 0 rather than a real value. The result is an energy column that is half real
data and half zeros.

The lesson generalizes well beyond this file: **always look at your data before
computing statistics on it.** A naive `colvar["ene"].mean()` here would be off by
roughly a factor of two, and nothing in the calculation would warn you. In Section 9 you
will filter these rows out before using the energy.

---
## Section 8 — Cross-Validating MDAnalysis Against PLUMED

You now have two completely independent computations of the same six dihedral angles:

- **`dihedrals_mda`** — computed by you, from xyz coordinates in `finalrun.xtc`, using
  the torsion formula from Section 6
- **`colvar`** — computed by PLUMED, live during the simulation, from GROMACS's internal
  coordinates

If both are right, they must agree. This kind of check is one of the most valuable habits
in computational work: **two independent paths to the same number**. When they agree you
gain real confidence; when they disagree you have found a bug, and the disagreement tells
you where to look.

### Matching on time, not row index

The two datasets have different sampling rates (20 ps vs 0.1 ps), so row *i* of one is not
row *i* of the other. We match on the `time` column instead. Since every trajectory time
(0, 20, 40, … ps) also appears exactly in the COLVAR file, `np.searchsorted` finds each
one cleanly:

```python
idx = np.searchsorted(colvar["time"].values, times_mda)
matched = colvar.iloc[idx].reset_index(drop=True)
```

### Comparing angles correctly

Angles are **periodic**: −179° and +179° are 2° apart, not 358° apart. Subtracting them
naively gives a huge fake error every time an angle crosses the ±180° seam. The standard
fix wraps the difference back into the range −π…π:

```python
diff = (a - b + np.pi) % (2*np.pi) - np.pi
```

Always use this when comparing or averaging angles.

In [ ]:
# Line up the two datasets on the time axis.

idx = np.searchsorted(colvar["time"].values, times_mda)
matched = colvar.iloc[idx].reset_index(drop=True)

time_error = np.abs(matched["time"].values - times_mda).max()
print(f"Largest time mismatch after alignment: {time_error} ps")
print(f"Matched {len(matched)} PLUMED rows to {len(times_mda)} trajectory frames")
print()
print("Frame 0 side by side:")
print(f"{'angle':>6}  {'MDAnalysis':>12}  {'PLUMED':>12}")
for name in dihedrals_mda:
    print(f"{name:>6}  {dihedrals_mda[name][0]:12.5f}  {matched[name].iloc[0]:12.5f}")


#### Worked Example 8.1
Quantify the agreement between your dihedrals and PLUMED's.

For each of the six angles, compute the periodic difference between `dihedrals_mda[name]`
and `matched[name].values`, then take the maximum absolute value. Store the results in
`max_dihedral_error`, a dictionary mapping angle name → max absolute error in radians.

Also set `worst_error` to the largest value across all six angles.

Remember to use the periodic wrapping formula — without it, any frame where an angle sits
near ±180° will produce a spurious error of about 2π.

In [ ]:
# Worked Example 8.1
max_dihedral_error = {}

for name, mda_vals in dihedrals_mda.items():
    plumed_vals = matched[name].values
    diff = (mda_vals - plumed_vals + np.pi) % (2 * np.pi) - np.pi
    max_dihedral_error[name] = float(np.abs(diff).max())

worst_error = max(max_dihedral_error.values())

print(f"{'angle':>6}  {'max error (rad)':>16}  {'max error (deg)':>16}")
for name, err in max_dihedral_error.items():
    print(f"{name:>6}  {err:16.6f}  {np.degrees(err):16.4f}")
print(f"\nWorst disagreement across all six angles: {worst_error:.6f} rad "
      f"({np.degrees(worst_error):.3f}°)")


In [ ]:
# TEST 8.1
assert isinstance(max_dihedral_error, dict), "max_dihedral_error should be a dictionary"
assert set(max_dihedral_error.keys()) == set(dihedrals_mda.keys()), "Need an entry for all six angles"
assert worst_error < 0.05, (
    f"Worst error is {worst_error:.4f} rad — too large. If it is near 2*pi (~6.28), "
    "you forgot the periodic wrapping. If it is near pi, check your atom ordering.")
assert worst_error > 1e-6, "Agreement this perfect is suspicious — are you comparing PLUMED to itself?"
print(f"✓  All six dihedrals agree with PLUMED to within {np.degrees(worst_error):.2f}°")


Agreement to about 1° across 324 frames and six angles is exactly what we should expect
— and the residual is not sloppiness on either side. GROMACS's `.xtc` format is
**lossy-compressed**: it stores coordinates to about 0.001 nm (0.01 Å) precision to keep
file sizes manageable. PLUMED computed its angles from the full double-precision
coordinates in memory during the run, while you computed yours from the rounded values
written to disk. A 0.01 Å position error on bond vectors a few ångströms long propagates
to roughly this much angular error.

So the answer is not "they match perfectly" but something more useful: **they match to
within the precision of the file format**, and we can explain the remaining difference
from first principles.

#### Worked Example 8.2 — When definitions disagree

Now try the same check on the radius of gyration, and you will find it *fails*.

Compute `rg_mda_nm` — your unwrapped $R_g$ time series from Practice 5.2, converted from
ångströms to nanometers (divide by 10) — then compare it to `matched["rg"].values` and
store the mean absolute difference as `rg_error_allatom`.

It will be large. The reason is that PLUMED's `rg` was **not defined over all protein
atoms**. Your job is to find which selection PLUMED actually used. Try recomputing $R_g$
over the **Cα atoms only** across the trajectory, store it as `rg_ca_nm`, and compute
`rg_error_ca`.

> Reminder: unwrap each frame before computing, and note that `ca_atoms` is already
> defined from Section 3.

In [ ]:
# Worked Example 8.2
rg_mda_nm = rg_unwrapped / 10          # from Practice 5.2, Å -> nm
plumed_rg = matched["rg"].values

rg_error_allatom = float(np.abs(rg_mda_nm - plumed_rg).mean())

# Recompute using only the alpha carbons
rg_ca_list = []
for ts in u.trajectory:
    protein.unwrap(compound='fragments')
    rg_ca_list.append(ca_atoms.radius_of_gyration() / 10)

rg_ca_nm = np.array(rg_ca_list)
rg_error_ca = float(np.abs(rg_ca_nm - plumed_rg).mean())

print(f"Mean |difference| using ALL protein atoms : {rg_error_allatom:.6f} nm")
print(f"Mean |difference| using Cα atoms only     : {rg_error_ca:.6f} nm")
print(f"\nImprovement factor: {rg_error_allatom / rg_error_ca:.0f}x")


In [ ]:
# TEST 8.2
assert rg_error_allatom > 0.05, "The all-atom comparison should disagree substantially with PLUMED"
assert len(rg_ca_nm) == len(u.trajectory), "rg_ca_nm should have one value per frame"
assert rg_error_ca < 0.001, (
    f"Cα-only Rg should match PLUMED to better than 0.001 nm, got {rg_error_ca:.6f}. "
    "Did you unwrap each frame before computing?")
print(f"✓  All-atom Rg is off by {rg_error_allatom:.4f} nm, but Cα-only Rg matches "
      f"PLUMED to {rg_error_ca:.6f} nm")
print("   PLUMED's GYRATION was defined over the alpha carbons.")

plt.figure(figsize=(7, 3))
plt.plot(times_mda / 1000, plumed_rg, '-', lw=1.5, label="PLUMED rg", alpha=0.8)
plt.plot(times_mda / 1000, rg_ca_nm, '--', lw=1, label="MDAnalysis, Cα only")
plt.plot(times_mda / 1000, rg_mda_nm, ':', lw=1, label="MDAnalysis, all atoms")
plt.xlabel("Time (ns)")
plt.ylabel("Radius of gyration (nm)")
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()


This is the most important lesson in the notebook, and it is worth sitting with.

Both numbers were *computed correctly*. Neither MDAnalysis nor PLUMED made an arithmetic
error. They disagreed because they were answering **different questions**: "how spread out
are all 73 atoms of this peptide?" versus "how spread out are its four alpha carbons?"
Both are legitimate definitions of "radius of gyration," and the phrase alone does not
tell you which one you have.

A collective variable is not just a formula — it is a formula **plus an atom selection**,
and the selection lives in the PLUMED input file, not in the COLVAR output. When you
inherit someone's simulation data, or return to your own a year later, the CV definitions
are metadata you must go find. The number in the file will not tell you, and it will not
complain when you assume wrong.

Notice also *how* we diagnosed it: the all-atom and PLUMED curves in the plot are
correlated but offset, which is the signature of a definitional mismatch rather than a
bug. A genuine bug usually produces noise or nonsense, not a consistent offset.

---
## Section 9 — Visualizing the OPES Multithermal Landscape

Now that we trust both datasets, we can use the COLVAR file's much denser sampling —
65,133 points instead of 324 — to look at the conformational landscape properly.

### The Ramachandran plot

Plotting φ against ψ for a residue gives a **Ramachandran plot**, the standard map of
protein backbone conformation. Regions of this map correspond to secondary structures you
have seen in biology: α-helix around (−60°, −45°), β-sheet around (−120°, +130°). Empty
regions are conformations that are sterically forbidden — the atoms would clash.

### What the bias column tells us

Recall from Section 7 that `opes.bias` records how much artificial energy OPES added at
each step. Two things follow:

1. **The bias must be filtered the same way the energy was.** In Practice 7.2 you found
   that half the rows have `ene` exactly zero. Because OPES computes its bias *from* the
   energy, those same rows carry a meaningless bias too. Any analysis of `ene` or
   `opes.bias` must drop them first.

2. **The bias lets us undo the bias.** Because we know exactly how much help each
   configuration received, we can reweight the biased samples back to what an ordinary
   300 K simulation would have produced. Each frame gets a statistical weight

   $$w_i \propto e^{\,V_{\text{bias},i} / k_B T}$$

   where $k_B T \approx 2.494$ kJ/mol at 300 K. Configurations that were pushed hard
   (large negative bias) get small weights; configurations sampled with little help count
   nearly fully. In practice we subtract the maximum bias before exponentiating, which
   cancels out of the normalized weights and prevents numerical overflow.

The sections below are **read-and-run** — study them and modify them for your own systems.

In [ ]:
# --- Ramachandran plot: trajectory sampling vs PLUMED sampling ---
# Residue 3 (tyrosine) has both a phi and a psi, so we can map it.

fig, axes = plt.subplots(1, 2, figsize=(10, 4.2), sharex=True, sharey=True)

axes[0].plot(np.degrees(dihedrals_mda["phi3"]), np.degrees(dihedrals_mda["psi3"]),
             '.', ms=4, alpha=0.6, color='crimson')
axes[0].set_title(f"From the trajectory\n{len(times_mda)} frames (every 20 ps)")

axes[1].hexbin(np.degrees(colvar["phi3"]), np.degrees(colvar["psi3"]),
               gridsize=60, bins='log', cmap='viridis',
               extent=(-180, 180, -180, 180))
axes[1].set_title(f"From the COLVAR file\n{len(colvar):,} points (every 0.1 ps)")

for ax in axes:
    ax.set_xlabel("φ₃ (Tyr) / degrees")
    ax.set_xlim(-180, 180)
    ax.set_ylim(-180, 180)
    ax.axhline(0, color='grey', lw=0.5)
    ax.axvline(0, color='grey', lw=0.5)
axes[0].set_ylabel("ψ₃ (Tyr) / degrees")

plt.tight_layout()
plt.show()

print("Same simulation, same residue — but the saved trajectory shows scattered dots")
print("while the COLVAR file resolves the actual shape of the populated basins.")


In [ ]:
# --- Filtering the artifact, then examining the bias ---

real = colvar[colvar["ene"] != 0.0].reset_index(drop=True)

print(f"Rows before filtering : {len(colvar):,}")
print(f"Rows after filtering  : {len(real):,}")
print()
print("Effect of the artifact on simple statistics:")
print(f"  mean energy, unfiltered : {colvar['ene'].mean():12,.0f} kJ/mol   <- meaningless")
print(f"  mean energy, filtered   : {real['ene'].mean():12,.0f} kJ/mol")
print()
print(f"  bias std dev, unfiltered: {colvar['opes.bias'].std():12,.0f} kJ/mol   <- meaningless")
print(f"  bias std dev, filtered  : {real['opes.bias'].std():12,.0f} kJ/mol")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))

axes[0].plot(real["time"] / 1000, real["opes.bias"], lw=0.4, color='steelblue')
axes[0].set_xlabel("Time (ns)")
axes[0].set_ylabel("OPES bias (kJ/mol)")
axes[0].set_title("Bias deposited over time")

axes[1].hist(real["ene"], bins=80, color='darkorange', edgecolor='none')
axes[1].set_xlabel("Potential energy (kJ/mol)")
axes[1].set_ylabel("Count")
axes[1].set_title("Sampled energy distribution")

plt.tight_layout()
plt.show()


In [ ]:
# --- Reweighting back to unbiased 300 K behaviour ---
# Use only the well-equilibrated portion: OPES needs time to build up its bias,
# and the first ~2 ns is still filling in the landscape.

kB = 8.314462618e-3          # kJ/(mol K)
T = 300.0                    # K
kT = kB * T                  # ~2.494 kJ/mol

late = real[real["time"] > 2000.0].reset_index(drop=True)

bias = late["opes.bias"].values
# Subtract the max before exponentiating: prevents overflow and cancels in the
# normalisation, so it does not change the answer.
weights = np.exp((bias - bias.max()) / kT)
weights /= weights.sum()

# Effective sample size: how many *independent* unbiased samples these are worth
ess = 1.0 / np.sum(weights**2)

print(f"Frames used             : {len(late):,}  (t > 2 ns)")
print(f"Effective sample size    : {ess:,.0f}")
print(f"Efficiency               : {100 * ess / len(late):.1f}%")
print()

rg_vals = late["rg"].values
print(f"Rg, biased average       : {rg_vals.mean():.4f} nm")
print(f"Rg, reweighted to 300 K  : {np.sum(rg_vals * weights):.4f} nm")

fig, ax = plt.subplots(figsize=(6, 3.4))
bins = np.linspace(rg_vals.min(), rg_vals.max(), 60)
ax.hist(rg_vals, bins=bins, density=True, alpha=0.5,
        label="biased (as sampled)", color='steelblue')
ax.hist(rg_vals, bins=bins, weights=weights, density=True, alpha=0.5,
        label="reweighted to 300 K", color='crimson')
ax.set_xlabel("Radius of gyration (nm)")
ax.set_ylabel("Probability density")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()


### Reading that last result honestly

The effective sample size is worth dwelling on. We started with tens of thousands of
biased frames, and after reweighting they are worth only a few hundred *independent*
unbiased samples — an efficiency of well under one percent.

That is not a mistake in the calculation; it is the fundamental trade-off of enhanced
sampling. OPES multithermal bought us barrier crossings that an ordinary 300 K run would
essentially never have shown, and it paid for them in statistical weight. A handful of
configurations sampled at low bias dominate the reweighted average, while the vast
majority of frames contribute almost nothing.

The practical consequences:

- **Reweighted averages from this run are indicative, not precise.** Quoting the
  reweighted $R_g$ to four decimal places would be overselling it.
- **Always report the effective sample size** alongside any reweighted quantity. A
  reweighted average without an ESS is an unfalsifiable number.
- **Longer runs help, but so do better collective variables.** If the bias were applied
  along coordinates that described the peptide's slow motions more directly, the same
  simulation time would yield far better statistics.

That last point is exactly where this track goes next.

---
## Summary and Next Steps

You have worked through a complete MDAnalysis + PLUMED workflow for an enhanced-sampling
peptide simulation:

1. **Loaded** a `Universe` from GROMACS `.tpr` + `.xtc` files and confirmed the system
2. **Selected** atom subsets — protein, backbone, Cα, sidechains, water
3. **Derived** the 3D distance formula from the Pythagorean theorem and saw why
   **periodic boundary conditions** make naive distances wrong up to ~20% of the time
   in this very trajectory
4. **Computed radius of gyration by hand** from a mass-weighted sum, and diagnosed a
   real "broken molecule" PBC artifact that inflated it sevenfold in some frames
5. **Computed backbone dihedral angles from raw xyz coordinates**, connecting the
   torsion-angle formula to the Newman projections you learned in organic chemistry
6. **Parsed a PLUMED COLVAR file** and learned to recognize a real GROMACS/PLUMED
   energy-logging artifact (exact-zero `ene` values on alternating rows)
7. **Cross-validated** MDAnalysis-computed geometry against PLUMED's own collective
   variables — and found they agree to 4–5 significant figures
8. **Visualized** the free-energy-style landscape that OPES multithermal sampling
   reveals, at far higher time resolution than the saved trajectory alone provides

### What's next

This notebook treated the PLUMED collective variables (dihedral angles, radius of
gyration) as **hand-picked, human-chosen** descriptions of the peptide's shape. A
follow-up notebook will introduce **mlcolvar**, a library for **learning** collective
variables directly from simulation data with neural networks (Deep-LDA, Deep-TICA),
rather than picking them by hand — building on the coordinate math and PLUMED-file
literacy you developed here.
